# Sketch2Clothes — Train on Kaggle

**FashionSD-X pipeline:** SD 1.5 + LoRA (stage 1) + ControlNet sketch (stage 2)

1. Settings → Accelerator → **GPU T4 x2**
3. Run all cells
2. Settings → Internet → **ON**

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

### Recaption dataset (Florence-2-large — replaces broken BLIP captions)

Copy to a code cell and run (**GPU T4**, Internet ON). ~5270 ảnh ≈ 1–3 giờ.

**Quan trọng**
- Pin `transformers==4.44.2` (tránh lỗi `forced_bos_token_id`)
- Script patch bỏ `flash_attn` khỏi import check Florence (không cần cài flash-attn; load `eager`/`sdpa`)
- Ghi ra `/kaggle/working/...` — không ghi `/kaggle/input`

```python
!git clone https://github.com/lqb464/SketClothes.git
%cd SketClothes
!pip install -q -r requirements.txt
!pip install -q "transformers==4.44.2" timm einops

!python /kaggle/working/SketClothes/training/utils/generate_captions.py \
  --photos_dir /kaggle/input/datasets/lqb464/sketclothes-data/data/png/photos \
  --captions_file /kaggle/working/captions.json \
  --model_id microsoft/Florence-2-large \
  --batch_size 1 \
  --max_new_tokens 64 \
  --force \
  --save_txt \
  --txt_out_dir /kaggle/working/captions

!zip -r /kaggle/working/captions.zip /kaggle/working/captions /kaggle/working/captions.json
```

Download `captions.zip` từ Output.

Chỉ sửa caption lỗi: bỏ `--force`, thêm `--fix_bad_only`.

**Cần push repo** trước khi clone lại (fix nằm trong `generate_captions.py`).

If want to train Stable Diffusion + ControlNet, copy this to next cell and run:

```python
!git clone https://github.com/lqb464/SketClothes.git
%cd SketClothes
!pip install -q -r requirements.txt

os.environ["LORA_PATH"] = "/kaggle/input/datasets/lqb464/sketclothes-data/pytorch_lora_weights.safetensors"
os.environ["OUTPUT_DIR"] = "/kaggle/working/outputs"
os.environ["DATASET_ID"] = "/kaggle/input/datasets/lqb464/sketclothes-data/data"

# Stage 1 + 2 + export test
!python training/train_all.py
```

Stage 1 only (LoRA)

```python
!git clone https://github.com/lqb464/SketClothes.git
%cd SketClothes
!pip install -q -r requirements.txt

os.environ["STAGE"] = "lora"
os.environ["OUTPUT_DIR"] = "/kaggle/working/outputs"
os.environ["DATASET_ID"] = "/kaggle/input/datasets/lqb464/sketclothes-data/data"
!python training/train_all.py
```

Stage 2 only (ControlNet) — no LoRA

```python
!git clone https://github.com/lqb464/SketClothes.git
%cd SketClothes
!pip install -q -r requirements.txt

os.environ["STAGE"] = "controlnet"
os.environ["OUTPUT_DIR"] = "/kaggle/working/outputs"
os.environ["DATASET_ID"] = "/kaggle/input/datasets/lqb464/sketclothes-data/data"
# do not set LORA_PATH (or point to a missing path) → trains against base SD UNet
!python training/train_all.py
```

Stage 2 only (ControlNet) — with existing LoRA

```python
!git clone https://github.com/lqb464/SketClothes.git
%cd SketClothes
!pip install -q -r requirements.txt

os.environ["STAGE"] = "controlnet"
os.environ["LORA_PATH"] = "/kaggle/input/datasets/lqb464/sketclothes-data/pytorch_lora_weights.safetensors"
os.environ["OUTPUT_DIR"] = "/kaggle/working/outputs"
os.environ["DATASET_ID"] = "/kaggle/input/datasets/lqb464/sketclothes-data/data"
!python training/train_all.py
```

If want to train pix2pix, copy this to next code cell
```python
!git clone -b experiment/pix2pix https://github.com/lqb464/SketClothes.git
%cd SketClothes
!pip install -q -r training/requirements.txt

!python training/train_pix2pix.py \
  --dataset_id /kaggle/input/datasets/lqb464/sketclothes-data/data \
  --output_dir /kaggle/working/outputs/pix2pix
```